# 1.2 — Forecast Accuracy

- **Propósito:** Comparar as previsões de demanda (forecast) com a demanda realizada, calculando métricas de acuracidade.
- **Entrada:** `parts_hdbk_sandbox.pr_demand.demand_analytical_base`, `parts_hdbk_sandbox.pr_forecast.refined_forecast_enriched`
- **Saída:** Análise de acuracidade do forecast
- **Chave:** `segment` + `main_material` + `forecast_month` · **Carga:** Sob demanda

In [0]:
from pyspark.sql import functions as F

In [0]:
# ---------------------------------------------------------------------------
# Parâmetros
# ---------------------------------------------------------------------------
CATALOG = "parts_hdbk_sandbox"

# Demand
DEMAND_TABLE = f"{CATALOG}.pr_demand.demand_analytical_base"

# Forecast
FORECAST_TABLE = f"{CATALOG}.pr_forecast.refined_forecast_enriched"

print(f"Demanda  : {DEMAND_TABLE}")
print(f"Forecast : {FORECAST_TABLE}")

In [0]:
# ---------------------------------------------------------------------------
# Leitura: base analítica de demanda
# ---------------------------------------------------------------------------
df_demand = spark.table(DEMAND_TABLE)
print(f"Demanda — linhas: {df_demand.count():,}")
print(f"Colunas: {df_demand.columns}")
display(df_demand.limit(5))

In [0]:
# ---------------------------------------------------------------------------
# Leitura: forecast enriched (refined, unpivot)
# ---------------------------------------------------------------------------
df_forecast = spark.table(FORECAST_TABLE)
print(f"Forecast — linhas: {df_forecast.count():,}")
print(f"Colunas: {df_forecast.columns}")
display(df_forecast.limit(5))

In [0]:
# ---------------------------------------------------------------------------
# Forecast agregado: apenas lag = 3, agrupado por segment + market +
# main_material + forecast_month, somando forecast_qty
# ---------------------------------------------------------------------------
df_forecast_agg = (
    df_forecast
    .filter(F.col("lag") == 3)
    .groupBy("segment", "market", "main_material", "forecast_month")
    .agg(F.sum("forecast_qty").alias("forecast_qty"))
)

print(f"Forecast (lag=3) agregado — linhas: {df_forecast_agg.count():,}")
display(df_forecast_agg.limit(5))

In [0]:
# ---------------------------------------------------------------------------
# Demanda agregada: data_ordem truncada no 1º dia do mês, agrupada por
# organizacao_vendas + canal_distribuicao + familia_produto + mês,
# somando quantidade
# ---------------------------------------------------------------------------
df_demand_agg = (
    df_demand
    .withColumn("mes_demanda", F.trunc("data_ordem", "month"))
    .groupBy("organizacao_vendas", "canal_distribuicao", "familia_produto", "mes_demanda")
    .agg(F.sum("quantidade").alias("quantidade"))
)

print(f"Demanda mensal agregada — linhas: {df_demand_agg.count():,}")
display(df_demand_agg.limit(5))

In [0]:
# ---------------------------------------------------------------------------
# Mapeamento canal_distribuicao → market para viabilizar o join
#   01 = Domestic, 02 = Export  (conforme README do projeto)
# ---------------------------------------------------------------------------
df_demand_mapped = df_demand_agg.withColumn(
    "market",
    F.when(F.col("canal_distribuicao") == "01", "Domestic")
     .when(F.col("canal_distribuicao") == "02", "Export")
     .otherwise(F.col("canal_distribuicao"))
)

# ---------------------------------------------------------------------------
# Join: forecast (lag=3) x demanda realizada
#   segment        ↔ organizacao_vendas
#   market         ↔ canal_distribuicao (mapeado)
#   main_material  ↔ familia_produto
#   forecast_month ↔ mes_demanda
# ---------------------------------------------------------------------------
df_accuracy = (
    df_forecast_agg.alias("f")
    .join(
        df_demand_mapped.alias("d"),
        (F.col("f.segment") == F.col("d.organizacao_vendas"))
        & (F.col("f.market") == F.col("d.market"))
        & (F.col("f.main_material") == F.col("d.familia_produto"))
        & (F.col("f.forecast_month") == F.col("d.mes_demanda")),
        "inner",
    )
    .select(
        F.col("f.segment"),
        F.col("f.market"),
        F.col("f.main_material"),
        F.col("f.forecast_month"),
        F.col("f.forecast_qty"),
        F.col("d.quantidade").alias("actual_qty"),
    )
)

# ---------------------------------------------------------------------------
# Métricas de acuracidade por linha
#   error          = forecast - realizado
#   abs_error      = |error|
#   abs_pct_error  = |error| / |realizado|  (MAPE unitário)
#   accuracy       = 1 - abs_pct_error
# ---------------------------------------------------------------------------
# df_accuracy = df_accuracy.withColumns({
#     "error": F.col("forecast_qty") - F.col("actual_qty"),
#     "abs_error": F.abs(F.col("forecast_qty") - F.col("actual_qty")),
#     "abs_pct_error": F.when(
#         F.col("actual_qty") != 0,
#         F.abs(F.col("forecast_qty") - F.col("actual_qty")) / F.abs(F.col("actual_qty"))
#     ),
#     "accuracy": F.when(
#         F.col("actual_qty") != 0,
#         1 - F.abs(F.col("forecast_qty") - F.col("actual_qty")) / F.abs(F.col("actual_qty"))
#     ),
# })

matched = df_accuracy.count()
print(f"Registros com match (inner join): {matched:,}")
display(df_accuracy.orderBy("segment", "main_material", "forecast_month").limit(10))